# Run MegaDetector on Colab (GPU) for CamTrap Lite

Detection is the only GPU-heavy step. This notebook runs **MegaDetector v5a** on a GPU runtime
and writes the standard **MegaDetector batch output JSON**. Load it into the app with:

```
python scripts/import_md_results.py --results results.json [--camera <id>]
```

**Before running:** *Runtime → Change runtime type → GPU*.

## 1. Install MegaDetector

In [ ]:
!pip install -q megadetector
import torch
print("GPU:", torch.cuda.is_available())

## 2. Get the images

Either upload a zip of images (option A) or point at a folder on Google Drive (option B).
Keep the same folder structure you uploaded to CamTrap Lite: the importer matches results
to images by relative path, falling back to the bare filename.

In [ ]:
# Option A: upload a zip from your computer
from google.colab import files
import zipfile, os

uploaded = files.upload()
zip_name = next(iter(uploaded))
IMAGE_DIR = "/content/images"
os.makedirs(IMAGE_DIR, exist_ok=True)
zipfile.ZipFile(zip_name).extractall(IMAGE_DIR)

In [ ]:
# Option B: use a folder on Google Drive instead (skip option A)
# from google.colab import drive
# drive.mount("/content/drive")
# IMAGE_DIR = "/content/drive/MyDrive/camtrap_images"

## 3. Run MegaDetector

`MDV5A` is downloaded automatically. `--output_relative_filenames` makes the paths in the
JSON relative to `IMAGE_DIR`; `--threshold 0.05` drops near-zero boxes the app ignores
anyway (they can't change triage).

In [ ]:
!python -m megadetector.detection.run_detector_batch MDV5A "$IMAGE_DIR" /content/results.json     --recursive --output_relative_filenames --threshold 0.05 --checkpoint_frequency 1000

## 4. Quick summary and download

In [ ]:
import json
r = json.load(open("/content/results.json"))
n = len(r["images"])
with_animal = sum(
    any(d["category"] == "1" and d["conf"] >= 0.2 for d in im.get("detections") or [])
    for im in r["images"]
)
print(f"{n} images, {with_animal} with an animal detection >= 0.2")
files.download("/content/results.json")